## Importing the libraries

In [13]:
import pandas as pd
import numpy as np
import keras
from ucimlrepo import fetch_ucirepo

In [ ]:
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets 
  
#Create the Dataframe
df = pd.DataFrame(X)
df['target'] = y

# metadata 
print(mushroom.metadata) 
  
# variable information 
print(mushroom.variables)

print(df.head())

{'uci_id': 73, 'name': 'Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/73/mushroom', 'data_url': 'https://archive.ics.uci.edu/static/public/73/data.csv', 'abstract': 'From Audobon Society Field Guide; mushrooms described in terms of physical characteristics; classification: poisonous or edible', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 8124, 'num_features': 22, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['poisonous'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1981, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5959T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family (pp. 500-525).  Each species is identified as definitely edible, definitely po

In [19]:
columns = df.columns
col_hot = []
col_bin = []

for col in columns:
    print(col,df[col].unique())
    if df[col].nunique() > 2:
        col_hot.append(col)
    else:
        col_bin.append(col)


#print(col_hot)
#print(col_bin)

cap-shape <StringArray>
['x', 'b', 's', 'f', 'k', 'c']
Length: 6, dtype: str
cap-surface <StringArray>
['s', 'y', 'f', 'g']
Length: 4, dtype: str
cap-color <StringArray>
['n', 'y', 'w', 'g', 'e', 'p', 'b', 'u', 'c', 'r']
Length: 10, dtype: str
bruises <StringArray>
['t', 'f']
Length: 2, dtype: str
odor <StringArray>
['p', 'a', 'l', 'n', 'f', 'c', 'y', 's', 'm']
Length: 9, dtype: str
gill-attachment <StringArray>
['f', 'a']
Length: 2, dtype: str
gill-spacing <StringArray>
['c', 'w']
Length: 2, dtype: str
gill-size <StringArray>
['n', 'b']
Length: 2, dtype: str
gill-color <StringArray>
['k', 'n', 'g', 'p', 'w', 'h', 'u', 'e', 'b', 'r', 'y', 'o']
Length: 12, dtype: str
stalk-shape <StringArray>
['e', 't']
Length: 2, dtype: str
stalk-root <StringArray>
['e', 'c', 'b', 'r', nan]
Length: 5, dtype: str
stalk-surface-above-ring <StringArray>
['s', 'f', 'k', 'y']
Length: 4, dtype: str
stalk-surface-below-ring <StringArray>
['s', 'f', 'y', 'k']
Length: 4, dtype: str
stalk-color-above-ring <Strin

## Refresh Import

In [49]:
import importlib
import data_cleaning_and_preprocessing

importlib.reload(data_cleaning_and_preprocessing)

from data_cleaning_and_preprocessing import DataProcessor

# Create an instance of the DataProcessor class
processor = DataProcessor(df)

## Handle Missing Data

In [51]:
processor.handle_missing(strategy='most_frequent',columns=['stalk-root'])
processor.get_df().isnull().sum()

cap-shape                   0
cap-surface                 0
cap-color                   0
bruises                     0
odor                        0
gill-attachment             0
gill-spacing                0
gill-size                   0
gill-color                  0
stalk-shape                 0
stalk-root                  0
stalk-surface-above-ring    0
stalk-surface-below-ring    0
stalk-color-above-ring      0
stalk-color-below-ring      0
veil-type                   0
veil-color                  0
ring-number                 0
ring-type                   0
spore-print-color           0
population                  0
habitat                     0
target                      0
dtype: int64

## One-Hot Encoding

In [55]:
col_bin = ['bruises','veil-type']
col_hot = columns.drop(col_bin).tolist()

print(col_hot)

# Preprocess the data
processor.encode_onehot(col_hot)

print(processor.get_df().head())


for col in col_bin:
    processor.map_binary(col)

print(processor.get_df().info())

['cap-shape', 'cap-surface', 'cap-color', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat', 'target']
  bruises veil-type  cap-shape_c  cap-shape_f  cap-shape_k  cap-shape_s  \
0       t         p          0.0          0.0          0.0          0.0   
1       t         p          0.0          0.0          0.0          0.0   
2       t         p          0.0          0.0          0.0          0.0   
3       t         p          0.0          0.0          0.0          0.0   
4       f         p          0.0          0.0          0.0          0.0   

   cap-shape_x  cap-surface_g  cap-surface_s  cap-surface_y  ...  \
0          1.0            0.0            1.0            0.0  ...   
1          1.0            0.0            1.0            0.0  ...  

## Variance-Threshold

In [56]:
# d = processor.get_df()
# print(d.shape)
# print(d.dtypes.value_counts())
# print("num cols:", d.select_dtypes(include=np.number).shape[1])
# print("first cols:", d.columns[:10])


processor.variance_threshold()

processor.get_df().columns

print(processor.get_df().info())

<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 95 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   bruises                     8124 non-null   float64
 1   cap-shape_c                 8124 non-null   float64
 2   cap-shape_f                 8124 non-null   float64
 3   cap-shape_k                 8124 non-null   float64
 4   cap-shape_s                 8124 non-null   float64
 5   cap-shape_x                 8124 non-null   float64
 6   cap-surface_g               8124 non-null   float64
 7   cap-surface_s               8124 non-null   float64
 8   cap-surface_y               8124 non-null   float64
 9   cap-color_c                 8124 non-null   float64
 10  cap-color_e                 8124 non-null   float64
 11  cap-color_g                 8124 non-null   float64
 12  cap-color_n                 8124 non-null   float64
 13  cap-color_p                 8124 non-null   

## Correlation Map

In [61]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# corr_matrix = processor.get_df().corr(numeric_only=True)
# print(corr_matrix.head())

# plt.figure(figsize=(12, 10))
# sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
# plt.title("Correlation Heatmap")
# plt.show()